# Ablation Grid over tl.sites

This 50-minute-track notebook is intentionally self-contained: it uses a seeded tiny PyTorch model, no downloads, and only public TorchLens APIs. The goal is to show the full workflow shape you would use on a larger model while keeping every cell runnable on CPU.


An ablation grid turns site discovery into a repeatable sweep. `tl.sites` records the intended sweep dimensions, and each concrete selector can be resolved and replayed independently.


In [ ]:
from __future__ import annotations

import math
from typing import Any

import matplotlib.pyplot as plt
import torch
from torch import nn

import torchlens as tl

SEED = 1608
torch.manual_seed(SEED)
torch.set_grad_enabled(False)


class TinyGridModel(nn.Module):
    """MLP with repeated operations for a small site sweep."""

    def __init__(self) -> None:
        """Initialize layers."""
        super().__init__()
        self.blocks = nn.ModuleList([nn.Linear(6, 6), nn.Linear(6, 6), nn.Linear(6, 6)])
        self.out = nn.Linear(6, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run repeated ReLU blocks and a readout."""
        for block in self.blocks:
            x = torch.relu(block(x))
        return self.out(x)

In [ ]:
model = TinyGridModel().eval()
x = torch.randn(5, 6)
clean = tl.log_forward_pass(model, x, vis_opt="none", intervention_ready=True, name="clean")
sweep = tl.sites(tl.func("relu"), modes=["zero", "scale"])
print(f"sweep entries: {len(sweep)}")
print([site.layer_label for site in clean.find_sites(tl.func("relu"))])

In [ ]:
rows: list[dict[str, Any]] = []
clean_output = clean.layer_list[-1].activation
for concrete_site in clean.find_sites(tl.func("relu")):
    for mode, hook in [("zero", tl.zero_ablate()), ("scale", tl.scale(0.25))]:
        edited = clean.fork(f"{mode}_{concrete_site.layer_label}")
        edited.attach_hooks(tl.label(concrete_site.layer_label), hook).replay()
        delta = torch.linalg.vector_norm(edited.layer_list[-1].activation - clean_output)
        rows.append({"site": concrete_site.layer_label, "mode": mode, "delta": float(delta)})

for row in rows:
    print(f"{row['site']:18s} {row['mode']:5s} delta {row['delta']:.4f}")

In [ ]:
site_labels = sorted({row["site"] for row in rows})
mode_labels = ["zero", "scale"]
grid = torch.zeros(len(site_labels), len(mode_labels))
for row in rows:
    grid[site_labels.index(row["site"]), mode_labels.index(row["mode"])] = row["delta"]

fig, ax = plt.subplots(figsize=(5, 3))
image = ax.imshow(grid.numpy(), cmap="cividis")
ax.set_xticks(range(len(mode_labels)))
ax.set_xticklabels(mode_labels)
ax.set_yticks(range(len(site_labels)))
ax.set_yticklabels(site_labels)
ax.set_title("Output delta by ablation site")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()